In [2]:
# ============================================================
# PHASE 6: WHAT-IF SIMULATOR
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# Project paths
project_path = os.path.abspath("..")
processed_path = os.path.join(project_path, "data", "processed")

# Load Phase 3 and Phase 5 datasets
connections = pd.read_csv(
    os.path.join(processed_path, "aircraft_connections.csv")
)

interventions = pd.read_csv(
    os.path.join(processed_path, "phase5_intervention_analysis.csv")
)

print("=" * 70)
print("PHASE 6: WHAT-IF SIMULATOR")
print("=" * 70)

print("\nPhase 3 connections:", connections.shape)
print("Phase 5 intervention data:", interventions.shape)

print("\nPhase 5 columns:")
print(interventions.columns.tolist())

PHASE 6: WHAT-IF SIMULATOR

Phase 3 connections: (479665, 35)
Phase 5 intervention data: (479665, 72)

Phase 5 columns:
['PREV_FLIGHT_ID', 'PREV_TAIL_NUM', 'PREV_CARRIER', 'PREV_FLIGHT_NUM', 'PREV_FL_DATE', 'PREV_ORIGIN', 'PREV_DEST', 'PREV_DEP_UTC', 'PREV_ARR_UTC', 'PREV_CRS_DEP_UTC', 'PREV_CRS_ARR_UTC', 'PREV_DEP_DELAY', 'PREV_ARR_DELAY', 'NEXT_FLIGHT_ID', 'NEXT_TAIL_NUM', 'NEXT_CARRIER', 'NEXT_FLIGHT_NUM', 'NEXT_FL_DATE', 'NEXT_ORIGIN', 'NEXT_DEST', 'NEXT_DEP_UTC', 'NEXT_ARR_UTC', 'NEXT_CRS_DEP_UTC', 'NEXT_CRS_ARR_UTC', 'NEXT_DEP_DELAY', 'NEXT_ARR_DELAY', 'SAME_AIRCRAFT', 'AIRPORT_CONTINUITY', 'ACTUAL_TIME_VALID', 'ACTUAL_TURNAROUND_MIN', 'SCHEDULED_TURNAROUND_BUFFER_MIN', 'BUFFER_CONSUMED_MIN', 'SHORT_TURNAROUND_FLAG', 'CONNECTION_VALID', 'CONNECTION_REJECTION_REASON', 'PREVIOUS_DELAYED', 'NEXT_DELAYED', 'POTENTIAL_PROPAGATION', 'PREVIOUS_DELAY_CATEGORY', 'BUFFER_CONSUMPTION_CATEGORY', 'CALCULATED_BUFFER_CONSUMED', 'BUFFER_DIFFERENCE', 'TURNAROUND_CATEGORY', 'ESTIMATED_REMAINING_BU

In [3]:
# ============================================================
# 1. PREPARE SIMULATION DATA
# ============================================================

required_columns = [
    "PREV_TAIL_NUM",
    "PREV_FLIGHT_ID",
    "NEXT_FLIGHT_ID",
    "PREV_ARR_DELAY",
    "NEXT_DEP_DELAY",
    "SCHEDULED_TURNAROUND_BUFFER_MIN"
]

missing_columns = [
    col for col in required_columns
    if col not in interventions.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

sim_data = interventions.copy()

# Remove records with missing identifiers
sim_data = sim_data.dropna(
    subset=[
        "PREV_TAIL_NUM",
        "PREV_FLIGHT_ID",
        "NEXT_FLIGHT_ID"
    ]
).copy()

# Convert identifiers into consistent strings
for col in [
    "PREV_TAIL_NUM",
    "PREV_FLIGHT_ID",
    "NEXT_FLIGHT_ID"
]:
    sim_data[col] = sim_data[col].astype(str)

# Convert numerical fields
numeric_columns = [
    "PREV_ARR_DELAY",
    "NEXT_DEP_DELAY",
    "SCHEDULED_TURNAROUND_BUFFER_MIN"
]

for col in numeric_columns:
    sim_data[col] = pd.to_numeric(
        sim_data[col],
        errors="coerce"
    )

sim_data = sim_data.dropna(
    subset=numeric_columns
).copy()

# Remove invalid negative turnaround buffers
sim_data = sim_data[
    sim_data["SCHEDULED_TURNAROUND_BUFFER_MIN"] >= 0
].copy()

# Optional next-flight arrival delay
if "NEXT_ARR_DELAY" in sim_data.columns:
    sim_data["NEXT_ARR_DELAY"] = pd.to_numeric(
        sim_data["NEXT_ARR_DELAY"],
        errors="coerce"
    )

print("Usable simulation connections:", len(sim_data))
print("Unique aircraft:", sim_data["PREV_TAIL_NUM"].nunique())

print("\nMissing values:")
display(sim_data[required_columns].isnull().sum())

print("\nSample records:")
display(sim_data[required_columns].head())

Usable simulation connections: 479665
Unique aircraft: 5605

Missing values:


PREV_TAIL_NUM                      0
PREV_FLIGHT_ID                     0
NEXT_FLIGHT_ID                     0
PREV_ARR_DELAY                     0
NEXT_DEP_DELAY                     0
SCHEDULED_TURNAROUND_BUFFER_MIN    0
dtype: int64


Sample records:


,PREV_TAIL_NUM,PREV_FLIGHT_ID,NEXT_FLIGHT_ID,PREV_ARR_DELAY,NEXT_DEP_DELAY,SCHEDULED_TURNAROUND_BUFFER_MIN
0,N838SK,FL378922,FL378923,210.0,252.0,3.0
1,N679CA,FL290284,FL290285,112.0,126.0,18.0
2,N421YX,FL187090,FL187091,91.0,89.0,30.0
3,N761ND,FL326257,FL326258,125.0,164.0,21.0
4,N533AE,FL228771,FL228772,66.0,78.0,30.0


In [4]:
# ============================================================
# 2. SELECT AN AIRCRAFT ROTATION
# ============================================================

# Find the available intervention ranking column
score_candidates = [
    "PRIORITY_SCORE",
    "OPERATIONAL_RISK_SCORE",
    "INTERVENTION_SCORE"
]

score_column = next(
    (col for col in score_candidates if col in sim_data.columns),
    None
)

if score_column:
    sim_data[score_column] = pd.to_numeric(
        sim_data[score_column],
        errors="coerce"
    ).fillna(0)

    sim_data = sim_data.sort_values(
        score_column,
        ascending=False
    )
else:
    # Fallback: choose connections with the largest incoming delay
    sim_data = sim_data.sort_values(
        "PREV_ARR_DELAY",
        ascending=False
    )

# Build a lookup to follow the same aircraft's next connection
connection_lookup = {}

for idx, row in sim_data.iterrows():
    key = (
        row["PREV_TAIL_NUM"],
        row["PREV_FLIGHT_ID"]
    )

    # Keep the first matching outgoing connection
    if key not in connection_lookup:
        connection_lookup[key] = idx

# Select the highest-ranked starting connection
start_index = sim_data.index[0]

start_row = sim_data.loc[start_index]

aircraft = start_row["PREV_TAIL_NUM"]

print("SELECTED AIRCRAFT")
print("-----------------")
print("Aircraft:", aircraft)
print("Starting previous flight:", start_row["PREV_FLIGHT_ID"])
print("Starting next flight:", start_row["NEXT_FLIGHT_ID"])
print("Initial arrival delay:", start_row["PREV_ARR_DELAY"], "minutes")
print("Scheduled turnaround buffer:",
      start_row["SCHEDULED_TURNAROUND_BUFFER_MIN"], "minutes")

if score_column:
    print("Intervention score:", start_row[score_column])

# Trace the downstream chain
chain_indices = []
visited = set()

current_index = start_index
max_chain_length = 10

for _ in range(max_chain_length):

    row = sim_data.loc[current_index]

    if current_index in visited:
        break

    visited.add(current_index)
    chain_indices.append(current_index)

    current_flight = row["NEXT_FLIGHT_ID"]

    next_key = (
        aircraft,
        current_flight
    )

    next_index = connection_lookup.get(next_key)

    if next_index is None:
        break

    current_index = next_index

chain = sim_data.loc[chain_indices].copy().reset_index(drop=True)

print("\nDOWNSTREAM ROTATION")
print("-------------------")
print("Aircraft:", aircraft)
print("Connections traced:", len(chain))

display(
    chain[
        [
            "PREV_FLIGHT_ID",
            "NEXT_FLIGHT_ID",
            "PREV_ARR_DELAY",
            "SCHEDULED_TURNAROUND_BUFFER_MIN",
            "NEXT_DEP_DELAY"
        ]
    ]
)

SELECTED AIRCRAFT
-----------------
Aircraft: N838SK
Starting previous flight: FL378922
Starting next flight: FL378923
Initial arrival delay: 210.0 minutes
Scheduled turnaround buffer: 3.0 minutes
Intervention score: 90.0

DOWNSTREAM ROTATION
-------------------
Aircraft: N838SK
Connections traced: 10


,PREV_FLIGHT_ID,NEXT_FLIGHT_ID,PREV_ARR_DELAY,SCHEDULED_TURNAROUND_BUFFER_MIN,NEXT_DEP_DELAY
0,FL378922,FL378923,210.0,3.0,252.0
1,FL378923,FL378924,238.0,40.0,229.0
2,FL378924,FL378925,239.0,31.0,253.0
3,FL378925,FL378926,243.0,685.0,316.0
4,FL378926,FL378927,297.0,40.0,288.0
5,FL378927,FL378928,270.0,52.0,251.0
6,FL378928,FL378929,262.0,79.0,230.0
7,FL378929,FL378930,273.0,46.0,267.0
8,FL378930,FL378931,276.0,810.0,96.0
9,FL378931,FL378932,89.0,40.0,83.0


In [6]:
# ============================================================
# 3. SIMULATION FUNCTION
# ============================================================

def simulate_rotation(
    chain,
    initial_delay,
    extra_buffer=0,
    scenario_name="Baseline"
):

    results = []

    simulated_arrival_delay = float(initial_delay)

    for i, row in chain.iterrows():

        observed_incoming_delay = float(
            row["PREV_ARR_DELAY"]
        )

        observed_next_departure = float(
            row["NEXT_DEP_DELAY"]
        )

        scheduled_buffer = float(
            row["SCHEDULED_TURNAROUND_BUFFER_MIN"]
        )

        # Apply hypothetical additional buffer
        available_buffer = scheduled_buffer + extra_buffer

        # Estimate the observed delay carryover component
        observed_carryover = max(
            0,
            observed_incoming_delay - scheduled_buffer
        )

        # Estimate independent delay component
        independent_delay = max(
            0,
            observed_next_departure - observed_carryover
        )

        # Simulate delay carried into the next departure
        simulated_carryover = max(
            0,
            simulated_arrival_delay - available_buffer
        )

        simulated_departure_delay = (
            independent_delay + simulated_carryover
        )

        # Estimate next flight's arrival delay
        if (
            "NEXT_ARR_DELAY" in chain.columns
            and pd.notna(row.get("NEXT_ARR_DELAY"))
        ):

            observed_flight_delay_change = (
                float(row["NEXT_ARR_DELAY"])
                - observed_next_departure
            )

        else:

            # Assumption when next arrival delay is unavailable:
            # no additional recovery or worsening during flight
            observed_flight_delay_change = 0

        simulated_next_arrival = max(
            0,
            simulated_departure_delay
            + observed_flight_delay_change
        )

        results.append({

            "SCENARIO": scenario_name,

            "FLIGHT_NUMBER": i + 1,

            "PREV_FLIGHT_ID": row["PREV_FLIGHT_ID"],

            "NEXT_FLIGHT_ID": row["NEXT_FLIGHT_ID"],

            "OBSERVED_INCOMING_DELAY": observed_incoming_delay,

            "SIMULATED_INCOMING_DELAY": simulated_arrival_delay,

            "SCHEDULED_BUFFER": scheduled_buffer,

            "ADDITIONAL_BUFFER": extra_buffer,

            "AVAILABLE_BUFFER": available_buffer,

            "ESTIMATED_INDEPENDENT_DELAY": independent_delay,

            "SIMULATED_CARRYOVER": simulated_carryover,

            "SIMULATED_DEPARTURE_DELAY": simulated_departure_delay,

            "SIMULATED_ARRIVAL_DELAY": simulated_next_arrival

        })

        # Pass this flight's simulated arrival delay to the next connection
        simulated_arrival_delay = simulated_next_arrival

    return pd.DataFrame(results)

In [7]:
# ============================================================
# 4. SCENARIO CONFIGURATION
# ============================================================

INITIAL_DELAY_REDUCTION = 30

ADDITIONAL_TURNAROUND_BUFFER = 15

initial_delay = float(
    chain.iloc[0]["PREV_ARR_DELAY"]
)

reduced_initial_delay = max(
    0,
    initial_delay - INITIAL_DELAY_REDUCTION
)

print("SCENARIO SETTINGS")
print("-----------------")

print("Original initial delay:", initial_delay, "minutes")

print(
    "Initial delay after intervention:",
    reduced_initial_delay,
    "minutes"
)

print(
    "Additional turnaround buffer:",
    ADDITIONAL_TURNAROUND_BUFFER,
    "minutes"
)

SCENARIO SETTINGS
-----------------
Original initial delay: 210.0 minutes
Initial delay after intervention: 180.0 minutes
Additional turnaround buffer: 15 minutes


In [8]:
# ============================================================
# 5. RUN WHAT-IF SCENARIOS
# ============================================================

# Scenario 1: Baseline
baseline = simulate_rotation(
    chain,
    initial_delay=initial_delay,
    extra_buffer=0,
    scenario_name="Baseline"
)

# Scenario 2: Reduce initial delay
delay_reduction = simulate_rotation(
    chain,
    initial_delay=reduced_initial_delay,
    extra_buffer=0,
    scenario_name="Initial Delay Reduced"
)

# Scenario 3: Increase turnaround buffer
buffer_increase = simulate_rotation(
    chain,
    initial_delay=initial_delay,
    extra_buffer=ADDITIONAL_TURNAROUND_BUFFER,
    scenario_name="Buffer Increased"
)

# Scenario 4: Apply both interventions
combined = simulate_rotation(
    chain,
    initial_delay=reduced_initial_delay,
    extra_buffer=ADDITIONAL_TURNAROUND_BUFFER,
    scenario_name="Combined Intervention"
)

all_scenarios = pd.concat(
    [
        baseline,
        delay_reduction,
        buffer_increase,
        combined
    ],
    ignore_index=True
)

print("All scenarios completed.")

display(all_scenarios.head(12))

All scenarios completed.


,SCENARIO,FLIGHT_NUMBER,PREV_FLIGHT_ID,NEXT_FLIGHT_ID,OBSERVED_INCOMING_DELAY,SIMULATED_INCOMING_DELAY,SCHEDULED_BUFFER,ADDITIONAL_BUFFER,AVAILABLE_BUFFER,ESTIMATED_INDEPENDENT_DELAY,SIMULATED_CARRYOVER,SIMULATED_DEPARTURE_DELAY,SIMULATED_ARRIVAL_DELAY
0,Baseline,1,FL378922,FL378923,210.0,210.0,3.0,0,3.0,45.0,207.0,252.0,238.0
1,Baseline,2,FL378923,FL378924,238.0,238.0,40.0,0,40.0,31.0,198.0,229.0,239.0
2,Baseline,3,FL378924,FL378925,239.0,239.0,31.0,0,31.0,45.0,208.0,253.0,243.0
3,Baseline,4,FL378925,FL378926,243.0,243.0,685.0,0,685.0,316.0,0.0,316.0,297.0
4,Baseline,5,FL378926,FL378927,297.0,297.0,40.0,0,40.0,31.0,257.0,288.0,270.0
5,Baseline,6,FL378927,FL378928,270.0,270.0,52.0,0,52.0,33.0,218.0,251.0,262.0
6,Baseline,7,FL378928,FL378929,262.0,262.0,79.0,0,79.0,47.0,183.0,230.0,273.0
7,Baseline,8,FL378929,FL378930,273.0,273.0,46.0,0,46.0,40.0,227.0,267.0,276.0
8,Baseline,9,FL378930,FL378931,276.0,276.0,810.0,0,810.0,96.0,0.0,96.0,89.0
9,Baseline,10,FL378931,FL378932,89.0,89.0,40.0,0,40.0,34.0,49.0,83.0,72.0


In [9]:
# ============================================================
# 6. SCENARIO COMPARISON
# ============================================================

scenario_summary = (
    all_scenarios
    .groupby("SCENARIO")
    .agg(
        Total_Departure_Delay_Minutes=(
            "SIMULATED_DEPARTURE_DELAY", "sum"
        ),

        Average_Departure_Delay_Minutes=(
            "SIMULATED_DEPARTURE_DELAY", "mean"
        ),

        Maximum_Departure_Delay_Minutes=(
            "SIMULATED_DEPARTURE_DELAY", "max"
        ),

        Total_Connections=(
            "NEXT_FLIGHT_ID", "count"
        )
    )
    .reset_index()
)

baseline_total = scenario_summary.loc[
    scenario_summary["SCENARIO"] == "Baseline",
    "Total_Departure_Delay_Minutes"
].iloc[0]

scenario_summary["Delay_Minutes_Reduced"] = (
    baseline_total
    - scenario_summary["Total_Departure_Delay_Minutes"]
)

scenario_summary["Reduction_Percentage"] = np.where(
    baseline_total > 0,
    scenario_summary["Delay_Minutes_Reduced"]
    / baseline_total * 100,
    0
)

print("SCENARIO COMPARISON")
print("=" * 70)

display(scenario_summary.round(2))

SCENARIO COMPARISON


,SCENARIO,Total_Departure_Delay_Minutes,Average_Departure_Delay_Minutes,Maximum_Departure_Delay_Minutes,Total_Connections,Delay_Minutes_Reduced,Reduction_Percentage
0,Baseline,2265.0,226.5,316.0,10,0.0,0.00
1,Buffer Increased,2010.0,201.0,316.0,10,255.0,11.26
2,Combined Intervention,1920.0,192.0,316.0,10,345.0,15.23
3,Initial Delay Reduced,2175.0,217.5,316.0,10,90.0,3.97


In [10]:
# ============================================================
# 8. INTERVENTION IMPACT
# ============================================================

combined_total = scenario_summary.loc[
    scenario_summary["SCENARIO"] == "Combined Intervention",
    "Total_Departure_Delay_Minutes"
].iloc[0]

delay_reduction_total = scenario_summary.loc[
    scenario_summary["SCENARIO"] == "Initial Delay Reduced",
    "Total_Departure_Delay_Minutes"
].iloc[0]

buffer_total = scenario_summary.loc[
    scenario_summary["SCENARIO"] == "Buffer Increased",
    "Total_Departure_Delay_Minutes"
].iloc[0]

print("=" * 65)
print("INTERVENTION IMPACT SUMMARY")
print("=" * 65)

print(
    "\nDelay reduction scenario:",
    round(baseline_total - delay_reduction_total, 2),
    "delay minutes reduced"
)

print(
    "\nBuffer increase scenario:",
    round(baseline_total - buffer_total, 2),
    "delay minutes reduced"
)

print(
    "\nCombined intervention:",
    round(baseline_total - combined_total, 2),
    "delay minutes reduced"
)

if baseline_total > 0:

    print(
        "\nCombined reduction percentage:",
        round(
            (baseline_total - combined_total)
            / baseline_total * 100,
            2
        ),
        "%"
    )

else:

    print("\nBaseline delay is zero; percentage reduction is undefined.")

INTERVENTION IMPACT SUMMARY

Delay reduction scenario: 90.0 delay minutes reduced

Buffer increase scenario: 255.0 delay minutes reduced

Combined intervention: 345.0 delay minutes reduced

Combined reduction percentage: 15.23 %


In [11]:
# ============================================================
# 9. SAVE PHASE 6 RESULTS
# ============================================================

phase6_path = os.path.join(
    processed_path,
    "phase6"
)

os.makedirs(
    phase6_path,
    exist_ok=True
)

# Save selected rotation
chain.to_csv(
    os.path.join(
        phase6_path,
        "phase6_selected_rotation.csv"
    ),
    index=False
)

# Save all scenario results
all_scenarios.to_csv(
    os.path.join(
        phase6_path,
        "phase6_scenario_results.csv"
    ),
    index=False
)

# Save summary
scenario_summary.to_csv(
    os.path.join(
        phase6_path,
        "phase6_scenario_comparison.csv"
    ),
    index=False
)

print("PHASE 6 OUTPUTS SAVED")
print("---------------------")

print("1. phase6_selected_rotation.csv")
print("2. phase6_scenario_results.csv")
print("3. phase6_scenario_comparison.csv")

print("\nPhase 6 simulation completed.")

PHASE 6 OUTPUTS SAVED
---------------------
1. phase6_selected_rotation.csv
2. phase6_scenario_results.csv
3. phase6_scenario_comparison.csv

Phase 6 simulation completed.
